# Lessons 04-06: Search evaluation, Hit Rate, MRR, boost tuning

No LLM calls here, so we run on the **full** course ground truth (395 questions). Note: the course had 79 LLM Zoomcamp FAQ documents on 2026-05-29; today's FAQ has more, so the index is bigger and numbers differ.

In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/course/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")
len(ground_truth)

395

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)
len(documents)

161

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    return index.search(query, num_results=5, boost_dict=boost_dict)

## Relevance lists (lesson 04)

In [4]:
q = ground_truth[0]
doc_id = q["document"]
results = text_search(query=q["question"])
print(q["question"])
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

Is it okay to join the course late if I just found it now?
74eb249bbf == 74eb249bbf: True
0fab61eca2 == 74eb249bbf: False
63bc46be32 == 74eb249bbf: False
610ccb23c0 == 74eb249bbf: False
5cc511f85b == 74eb249bbf: False


In [5]:
from tqdm.auto import tqdm

def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])
    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))
    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance_total.append(compute_relevance(q, search_function))
    return relevance_total

relevance_total = compute_relevance_total(ground_truth[:15], text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

## Metrics (lesson 05)

In [6]:
def hit_rate(relevance):
    cnt = 0
    for line in relevance:
        if 1 in line:
            cnt = cnt + 1
    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0
    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break
    return total_score / len(relevance)

example = [[1,0,0,0,0],[0,1,0,0,0],[1,0,0,0,0],[0,0,0,0,0],[0,1,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[0,0,1,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0],[1,0,0,0,0]]
round(hit_rate(example), 3), round(mrr(example), 3)

(0.933, 0.822)

In [7]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)
    return {"hit_rate": hit_rate(relevance_total), "mrr": mrr(relevance_total)}

evaluate(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.620253164556962, 'mrr': 0.5427848101265824}

## Boost tuning (lesson 06)

In [8]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}
    return index.search(query, num_results=5, boost_dict=boost_dict)

for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(ground_truth, lambda query, boost=boost: search_boost(query, boost))
    print(f"boost={boost}: hit_rate={result['hit_rate']:.3f} mrr={result['mrr']:.3f}")

  0%|          | 0/395 [00:00<?, ?it/s]

boost=0.5: hit_rate=0.666 mrr=0.571


  0%|          | 0/395 [00:00<?, ?it/s]

boost=1.0: hit_rate=0.671 mrr=0.578


  0%|          | 0/395 [00:00<?, ?it/s]

boost=3.0: hit_rate=0.620 mrr=0.543


  0%|          | 0/395 [00:00<?, ?it/s]

boost=5.0: hit_rate=0.608 mrr=0.521


  0%|          | 0/395 [00:00<?, ?it/s]

boost=10.0: hit_rate=0.590 mrr=0.500


In [9]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {"question": question_boost, "section": section_boost, "answer": answer_boost}
    return index.search(query, num_results=5, boost_dict=boost_dict)

results = []
for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            result = evaluate(
                ground_truth,
                lambda query, qb=question_boost, ab=answer_boost, sb=section_boost: search_boosts(query, qb, ab, sb)
            )
            results.append({"question": question_boost, "answer": answer_boost, "section": section_boost,
                            "hit_rate": result["hit_rate"], "mrr": result["mrr"]})

df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

...

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

  0%|          | 0/395 [00:00<?, ?it/s]

,question,answer,section,hit_rate,mrr
18,2.0,4.0,0.1,0.724051,0.626624
3,1.0,2.0,0.1,0.724051,0.626624
19,2.0,4.0,0.2,0.724051,0.626624
35,5.0,10.0,0.5,0.724051,0.626624
33,5.0,10.0,0.1,0.724051,0.626287
34,5.0,10.0,0.2,0.724051,0.626160
4,1.0,2.0,0.2,0.721519,0.624473
20,2.0,4.0,0.5,0.718987,0.622405
6,1.0,4.0,0.1,0.726582,0.614599
7,1.0,4.0,0.2,0.726582,0.613840


In [10]:
def text_search(query):
    boost_dict = {"question": 1.0, "answer": 2.0, "section": 0.1}
    return index.search(query, num_results=5, boost_dict=boost_dict)

evaluate(ground_truth, text_search)

  0%|          | 0/395 [00:00<?, ?it/s]

{'hit_rate': 0.7240506329113924, 'mrr': 0.6266244725738397}

## Extra: fair comparison with the course numbers
16 of the 78 course ground-truth documents are no longer in today's FAQ, so their questions can never be a hit. Keep only questions whose document still exists.

In [11]:
current_ids = {d["id"] for d in documents}
gt_present = [r for r in ground_truth if r["document"] in current_ids]
print(len(gt_present), "of", len(ground_truth), "questions still have their document")

def text_search_course_default(query):
    return index.search(query, num_results=5, boost_dict={"question": 3.0, "section": 0.5})

for name, fn in [("question=3, section=0.5 (before tuning)", text_search_course_default), ("question=1, answer=2, section=0.1 (tuned)", text_search)]:
    r = evaluate(gt_present, fn)
    print(f"{name}: hit_rate={r['hit_rate']:.3f} mrr={r['mrr']:.3f}")

315 of 395 questions still have their document


  0%|          | 0/315 [00:00<?, ?it/s]

question=3, section=0.5 (before tuning): hit_rate=0.778 mrr=0.681


  0%|          | 0/315 [00:00<?, ?it/s]

question=1, answer=2, section=0.1 (tuned): hit_rate=0.908 mrr=0.786


## Extra: our Groq-generated sample questions

In [12]:
gt_groq = pd.read_csv("data/ground_truth-groq-sample.csv").to_dict(orient="records")
print(len(gt_groq), "questions")
evaluate(gt_groq, text_search)

50 questions


  0%|          | 0/50 [00:00<?, ?it/s]

{'hit_rate': 0.92, 'mrr': 0.7466666666666666}